# Fine-tune NLLB for English ⇄ Kikamba (Understand Kamba)

Runs on a **free GPU** in about 1.5–2 hours. Uses only existing public data (see `training/README.md`).

**Kaggle:** Settings (right panel) → *Accelerator* → **GPU T4 x2**, and *Internet* → **On** (needs a phone-verified account).
**Colab:** Runtime → Change runtime type → **T4 GPU**.

Then *Run all*. The last cells print a comparison of the original and the fine-tuned model.

In [ ]:
# 0. Check internet access (Kaggle: right panel -> Session options -> Internet: On; needs a phone-verified account)
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
    print("Internet OK")
except OSError:
    raise SystemExit("No internet. Kaggle: open the right panel -> Session options -> Internet: On "
                     "(verify your phone in Kaggle Settings if the switch is greyed out). Then Run all again.")

In [ ]:
# 1. Get the project code and data
!git clone --depth 1 -b training https://github.com/lawmaluki/understand-kamba.git
%cd understand-kamba
!pip install -q "transformers>=4.45" peft sacrebleu sentencepiece accelerate

In [ ]:
# 2. Check the GPU
import torch
assert torch.cuda.is_available(), "No GPU: turn on the T4 GPU in the notebook settings"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9), "GB")

## 3. Train and evaluate

Scores the original model, filters the web pairs, fine-tunes with LoRA, then scores the fine-tuned model on
**FLORES devtest** (formal text) and **10 held-out children's stories** (everyday text), in both directions.

To go faster, use `--model facebook/nllb-200-distilled-600M`.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python training/finetune_nllb.py --model facebook/nllb-200-distilled-1.3B --out nllb-kam-ft

## 4. Look at some translations from the held-out stories

In [ ]:
import json
d = json.load(open("nllb-kam-ft/test_outputs.json", encoding="utf-8"))
pairs, hyps = d["tests"]["stories"], d["outputs"]["stories/kam_to_en"]
for (kam, eng), hyp in list(zip(pairs, hyps))[:15]:
    print("KAMBA    :", kam); print("REFERENCE:", eng); print("MODEL    :", hyp); print()

## 5. (Optional) Upload the model to your Hugging Face account

Only if the table above shows the fine-tuned model is better on **both** test sets.
Use a token with *write* access (create a fresh one; never paste an old token into a shared notebook).
The repository is created **private**.

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste a write token when asked
repo = "lawmaluki/nllb-200-1.3B-kamba-ft"
api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path="nllb-kam-ft", repo_id=repo, ignore_patterns=["checkpoints/*"])
print("uploaded:", "https://huggingface.co/" + repo)